In [ ]:
# import modules 
import os 

from ipyleaflet import Map, basemaps
from ipywidgets import Layout

import numpy as np
import matplotlib.pyplot as plt

from pcr.builder import build_model
from dotenv import load_dotenv

# set up a seed 
np.random.seed(42)

# load API Key  
load_dotenv()
cds_api_key = os.getenv('CDS-API-KEY')

# coba hitung empirical rec rate

In [ ]:
# render map and zoom in to the area of interest
m = Map(
    basemap=basemaps.Esri.WorldImagery,
    scroll_wheel_zoom=True,
    center=(7.6031203577833315, 81.77331084939762),
    zoom=12,
    layout=Layout(width='800px', height='500px')
)

m

In [ ]:
# extract the map extent, run after the map fully rendered 
bbox = [m.west, m.south, m.east, m.north]
bbox

In [ ]:
# get wave data 
model_now = build_model(
    year_start=2020,      # simulation will start at 1st of January on `year_start`
    year_end=2119,        # simulation will end at 31st of December on `year_end`
    nr_simulation=100_000,# number of monte carlo simulation, you can always adjust it before run the simulation 
    bbox=bbox,
    cds_api_key=cds_api_key, 
    # rec rate 
    rec_rate=None, 
    calibrate=True
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(model_now.day, model_now.hs, label='Significant Wave Height (Hs)')
ax.axhline(y=model_now.hs.mean(), color='r', linestyle='--', label=f'Mean Hs: {model_now.hs.mean():.2f} m')

ax.legend()

fig.show()

In [ ]:
# recovery rate 
Hs_mean = model_now.hs.mean()
Ew = Hs_mean ** 2 / 16 * 1025 * 9.81
R = 0.0142 * Ew**0.35

print(f"Recovery Rate: {R:.2f} m/day")
print(f"Approximately equal to {R*365.25:.2f} m/year vs calibrated value {model_now.rec_rate:.2f} m/year")
print(f"Recovery time approximately equal to {2/R:.2f} days for average of low erosion group (2m of erosion)")
print(f"within {2/R:.2f} days coastline recovers approximately \u0394Sl")

In [ ]:
sorted_gap = sorted(model_now.detected_storm.gap)
exceedance = np.linspace(0, 100, len(sorted_gap))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(sorted_gap, exceedance, marker='o', linestyle='-', color='blue')
ax.axvline(x=15)

Estimated recovery rate is higher than calibrated (6-7 m/year)

implement to PCR framework 

In [ ]:
# simulate one simulation 
model_now.detect_storms()

model_now.prepare_simulation()
hss, durs, dirs, tps = model_now._generate_batch()

track_time, track_shoreline, storm_count = model_now._simulate_one(hss, durs, dirs, tps, 0)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(track_time, track_shoreline, label='Shoreline Position (m)')
# ax.axhline(y=model_now.hs.mean(), color='r', linestyle='--', label=f'Mean Hs: {model_now.hs.mean():.2f} m')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Shoreline Position (m)')

ax.legend()

fig.show()

## Simulate with limiting recovery to recovery rate 

In [ ]:
# simulate one 
from pcr import storm, slr, erosion, shoreline

In [ ]:
# simulate gap between storms 

storm_count = 0

synth_start = storm.gap_nhpp_thinning(
    T = model_now.t_days, 
    monthly_lambda = model_now.fitted_lambdas, 
    date_start = model_now.date_start, 
    duration=durs,
    start_storm=storm_count,
    day_to_month=model_now.day_to_month,
    day_to_year=model_now.day_to_year,
    day_to_fac=model_now.day_to_fac, 
    fac_lambda=model_now.fac_lambda
)

storm_count_end = storm_count + len(synth_start)

synth_hs, _, synth_duration, synth_tp, synth_end, synth_gap = storm.slice_synthetic_batch(
    hss, dirs, durs, tps, synth_start, storm_count, storm_count_end
)

In [ ]:
synth_slr = slr.vector_simulate_slrAR6(
    day_start=synth_start,
    rate_sl=model_now.rate_ar6,
    day_sl=model_now.days_ar6,
    wl0=model_now.wl0,
    scenario=model_now.ar6_scenario,
)

_, synth_erosion = erosion.vector_mendoza(
    hss=synth_hs,
    tps=synth_tp,
    durs=synth_duration,
    m=model_now.m,
    C1=model_now.c1,
    C2=model_now.c2,
)

# look up each storm's recovery rate for its day in one vectorized indexing
# op (day_to_rec_rate is precomputed once in prepare_simulation)
day_idx = np.clip(synth_start.astype(int), 0, len(model_now.day_to_rec_rate) - 1)
synth_recovery = shoreline.vector_calculate_recovery(
    gaps=synth_gap,
    rec_rate=model_now.day_to_rec_rate[day_idx],
)

synth_retreat = shoreline.vector_calculate_slr_retreat(
    slrs=synth_slr,
    m=model_now.m,
)

track_time, _, track_shoreline_position = shoreline.vector_track_shoreline(
    day_start=synth_start,
    day_end=synth_end,
    recovery=synth_recovery,
    retreat=synth_retreat,
    erosion=synth_erosion,
)


In [ ]:
Hs_mean = model_now.hs.mean()
Ew = Hs_mean ** 2 / 16 * 1025 * 9.81
R = 0.0142 * Ew**0.35
Tr = 2 / R
rec_max = model_now.rec_rate*Tr

In [ ]:
new_recovery = np.clip(synth_recovery, a_min=None, a_max=rec_max)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(synth_recovery, 'x', label='Original Recovery Rate')
ax.plot(new_recovery, 'o', color='orange', label='Clipped Recovery Rate')
ax.axhline(y=model_now.rec_rate*Tr)

## Calibrate based on the capping 

In [ ]:
# initialize model 
model_calibrate = build_model(
    year_start=2020,
    year_end=2119,
    nr_simulation=1000,
    nr_batch=1000,
    scenario='0',
    rec_rate= model_now.rec_rate, # initial? 
    # additional set up 
    bbox=bbox,
    cds_api_key=cds_api_key, 
)

In [ ]:
from scipy.optimize import minimize_scalar

DEFAULT_GRID_DAYS = np.arange(1, 366 * 100, 30)
model_calibrate.detect_storms()

def evaluate_rec_rate(rec_rate, model=model_calibrate, grid_days=DEFAULT_GRID_DAYS, seed=0):
    model.rec_rate = rec_rate

    np.random.seed(seed)  # same storm sequence for every candidate rec_rate

    # run simulation 
    model.prepare_simulation()

    sim_count = 0
    while sim_count < model.nr_simulation:
        print(f'progress: {sim_count}/{model.nr_simulation} simulations completed', end='\r')
        hss, durs, dirs, tps = model._generate_batch()
        storm_count = 0

        for _ in range(model.nr_batch):
            model.track_time[sim_count], model.track_shoreline[sim_count], storm_count = model._simulate_one_cap(hss, durs, dirs, tps, storm_count)
            sim_count += 1

        if sim_count >= model.nr_simulation:
            break
    
    aligned = np.empty((len(grid_days), len(model.track_time)))
    for i, (t, s) in enumerate(zip(model.track_time, model.track_shoreline)):
        aligned[:, i] = np.interp(grid_days, t, s)

    median_all = np.median(aligned, axis=1)
    return np.sum(np.abs(median_all))

result = minimize_scalar(
    evaluate_rec_rate,
    bounds=(1/365, R*2), 
    method='bounded',
    options={'xatol': 1e-6}
)

In [ ]:
print(f"Optimal Recovery Rate: {result.x:.6f} m/day ~ {result.x*365.25:.2f} m/year")

In [ ]:
# apply to model 
model_now.rec_rate = result.x

# apply with capped 

model_now.prepare_simulation()

sim_count = 0
next_progress = 10
while sim_count < model_now.nr_simulation:
    hss, durs, dirs, tps = model_now._generate_batch()
    storm_count = 0

    for _ in range(model_now.nr_batch):
        model_now.track_time[sim_count], model_now.track_shoreline[sim_count], storm_count = model_now._simulate_one_cap(hss, durs, dirs, tps, storm_count)

        row = model_now.compute_statistics(sim_count)
        model_now.shoreline_stats[:, sim_count] = row.flatten()

        if not model_now.keep_tracks:
            # free this simulation's tracks now rather than waiting for the
            # run to finish, so peak memory stays bounded by nr_batch rather
            # than nr_simulation
            model_now.track_time[sim_count] = None
            model_now.track_shoreline[sim_count] = None

        sim_count += 1

        progress = sim_count / model_now.nr_simulation * 100
        if progress >= next_progress:
            print(f'progress: {progress:.0f} %')
            next_progress += 10

        if sim_count >= model_now.nr_simulation:
            break

In [ ]:
# TODO: compare different cases 

Cases: 
1. constant recovery rate (unlimited)
2. step recovery rate (constant recovery within recovery time, then drop to zero)
3. add future storm (decreasing) 

In [ ]:
import pcr.visualisation as viz 

ax = viz.plot_exceedance(
    model=model_now,
)

ax.set_title('Current storm, with step recovery')
ax.axhline(y=1)